# Assignment 4: Agentic AI Application

A simple agentic system that reads a question, picks the right tool, calls it, and produces a final answer.

**Student:** AKINOLA, Lanre Olusegun  
**Student ID:** 12349031  
**Course:** 700.390 Advanced Topics in Neurocomputing

**Tools available:**
- `search_web()` — internet search via DuckDuckGo
- `convert_currency()` — live exchange rates via Frankfurter API
- `get_multiple_rates()` — compare several currencies at once
- `calculate()` — simple arithmetic

**Agent trace format:** `QUESTION → TOOL → OBSERVE → ANSWER → TIME`

In [ ]:
%%capture
!pip install ddgs requests
print("Packages ready")

## 1. Tool Functions

Each tool is a separate function. They all return a dict with a `success` key so the agent can handle errors cleanly.

In [ ]:
import re
import math
import time
import json
import os
import requests


# ============================================================
# TOOL 1: Web Search  (adapted from course search_api.py)
# ============================================================

def search_web(query, max_results=5):
    """Search the internet using DuckDuckGo."""
    try:
        from ddgs import DDGS
        with DDGS(timeout=10) as ddgs:
            raw = list(ddgs.text(query, max_results=max_results))
        if not raw:
            return {"success": False, "error": "No results found.", "results": []}
        results = [
            {"title": r.get("title", ""), "body": r.get("body", ""), "url": r.get("href", "")}
            for r in raw[:max_results]
        ]
        return {"success": True, "results": results}
    except Exception as e:
        return {"success": False, "error": str(e), "results": []}


# ============================================================
# TOOL 2: Currency Conversion  (Frankfurter API)
# ============================================================

def convert_currency(amount, from_currency, to_currency):
    """Convert an amount from one currency to another."""
    try:
        from_c, to_c = from_currency.upper(), to_currency.upper()
        url = f"https://api.frankfurter.app/latest?from={from_c}&to={to_c}"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        rate = data["rates"][to_c]
        return {
            "success": True,
            "from": from_c, "to": to_c,
            "rate": rate, "amount": amount,
            "converted": round(amount * rate, 4),
            "date": data.get("date", "")
        }
    except KeyError:
        return {"success": False, "error": f"Unknown currency: {to_currency}"}
    except Exception as e:
        return {"success": False, "error": str(e)}


def get_multiple_rates(base, targets):
    """Get several exchange rates at once for comparison."""
    try:
        symbols = ",".join([t.upper() for t in targets])
        url = f"https://api.frankfurter.app/latest?from={base.upper()}&to={symbols}"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return {"success": True, "base": data["base"], "rates": data["rates"], "date": data.get("date", "")}
    except Exception as e:
        return {"success": False, "error": str(e)}


# ============================================================
# TOOL 3: Calculator
# ============================================================

def calculate(expression):
    """Evaluate a simple arithmetic expression safely."""
    try:
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return {"success": True, "expression": expression, "result": round(float(result), 6)}
    except ZeroDivisionError:
        return {"success": False, "error": "Division by zero."}
    except Exception as e:
        return {"success": False, "error": str(e)}


print("Tools loaded: search_web, convert_currency, get_multiple_rates, calculate")

## 2. Agent Logic

The agent automatically picks the right tool using keyword matching. It does not ask the user.

In [ ]:
def select_tool(question):
    """
    Decide which tool is needed.
    Returns: 'search', 'currency', 'multi_currency', or 'currency_calculate'
    """
    q = question.lower()
    currency_words = ["eur", "usd", "gbp", "jpy", "chf", "currency", "currencies"]
    has_currency = any(w in q for w in currency_words)

    # Multi-step: get rate + compute an amount
    if has_currency and any(k in q for k in ["how much is", "how much are", "equals", "equal to"]):
        return "currency_calculate"

    # Compare multiple currencies
    if has_currency and any(k in q for k in ["compare", "vs", "versus", "and"]):
        found = re.findall(r'\b(eur|usd|gbp|jpy|chf)\b', q)
        if len(set(found)) >= 2:
            return "multi_currency"

    # Simple conversion
    if has_currency and any(k in q for k in ["convert", "exchange", "rate", "to usd", "to eur", "to gbp"]):
        return "currency"

    return "search"


def _parse_currencies(question):
    """Extract currency codes and the first number from a question."""
    q = question.upper()
    currencies = re.findall(r'\b(EUR|USD|GBP|JPY|CHF)\b', q)
    amounts = re.findall(r'\b(\d+(?:\.\d+)?)\b', q)
    amount = float(amounts[0]) if amounts else 1.0
    return amount, currencies


def run_agent(question):
    """
    Main agent function.
    Prints: QUESTION -> TOOL -> OBSERVE -> ANSWER -> TIME
    """
    print("=" * 65)
    print(f"QUESTION : {question}")

    t0 = time.time()
    tool_name = select_tool(question)
    print(f"TOOL     : {tool_name}")

    answer = ""
    tool_result = None

    # ---- currency ----
    if tool_name == "currency":
        amount, currencies = _parse_currencies(question)
        from_c = currencies[0] if len(currencies) > 0 else "EUR"
        to_c   = currencies[1] if len(currencies) > 1 else "USD"
        tool_result = convert_currency(amount, from_c, to_c)
        print(f"OBSERVE  : rate={tool_result.get('rate')}, converted={tool_result.get('converted')}, date={tool_result.get('date')}")
        if tool_result["success"]:
            answer = (f"{amount} {tool_result['from']} = {tool_result['converted']} {tool_result['to']}  "
                      f"(1 {tool_result['from']} = {tool_result['rate']} {tool_result['to']}, as of {tool_result['date']})")
        else:
            answer = f"Currency error: {tool_result['error']}"

    # ---- multi_currency ----
    elif tool_name == "multi_currency":
        _, currencies = _parse_currencies(question)
        base    = currencies[0] if currencies else "EUR"
        targets = list(dict.fromkeys(c for c in currencies[1:] if c != base)) or ["USD", "GBP"]
        tool_result = get_multiple_rates(base, targets)
        print(f"OBSERVE  : base={tool_result.get('base')}, rates={tool_result.get('rates')}")
        if tool_result["success"]:
            lines = [f"Live exchange rates for 1 {tool_result['base']} (date: {tool_result['date']}):"]
            for cur, rate in tool_result["rates"].items():
                lines.append(f"  1 {tool_result['base']} = {rate} {cur}")
            answer = "\n".join(lines)
        else:
            answer = f"Failed to get rates: {tool_result['error']}"

    # ---- currency_calculate (two steps) ----
    elif tool_name == "currency_calculate":
        _, currencies = _parse_currencies(question)
        base   = currencies[0] if currencies else "EUR"
        target = currencies[-1] if len(currencies) > 1 else "USD"
        # Extract the amount to convert (look for "how much is 250" pattern)
        how_much = re.search(r'how much (?:is|are)\s+(\d+(?:\.\d+)?)', question, re.IGNORECASE)
        if how_much:
            amount = float(how_much.group(1))
        else:
            all_nums = re.findall(r'\b(\d+(?:\.\d+)?)\b', question)
            amount = max(float(n) for n in all_nums) if all_nums else 1.0

        rate_result = convert_currency(1, base, target)
        print(f"OBSERVE  [step 1 - rate ] : 1 {base} = {rate_result.get('rate')} {target}")
        if rate_result["success"]:
            rate = rate_result["rate"]
            expr = f"{amount} * {rate}"
            calc_result = calculate(expr)
            print(f"OBSERVE  [step 2 - calc ] : {expr} = {calc_result.get('result')}")
            if calc_result["success"]:
                answer = (f"{amount} {base} = {calc_result['result']} {target}  "
                          f"(calculation: {amount} x {rate} = {calc_result['result']}, date: {rate_result['date']})")
            else:
                answer = f"Calculation error: {calc_result['error']}"
            tool_result = {"rate_step": rate_result, "calc_step": calc_result}
        else:
            answer = f"Rate error: {rate_result['error']}"
            tool_result = rate_result

    # ---- search ----
    else:
        tool_result = search_web(question)
        n = len(tool_result.get("results", []))
        print(f"OBSERVE  : {n} result(s) returned from DuckDuckGo")
        if tool_result["success"] and tool_result["results"]:
            snippets = [r["body"].strip() for r in tool_result["results"] if r.get("body")]
            combined = " ".join(snippets[:2])
            if len(combined) > 700:
                combined = combined[:700] + "..."
            sources = [r["url"] for r in tool_result["results"] if r.get("url")]
            answer = combined + (f"\n[Source: {sources[0]}]" if sources else "")
        else:
            answer = f"Search returned no results. {tool_result.get('error', '')}"

    elapsed = round(time.time() - t0, 2)
    display = answer if len(answer) <= 350 else answer[:350] + "..."
    print(f"ANSWER   : {display}")
    print(f"TIME     : {elapsed}s")
    print()

    return {"question": question, "tool": tool_name, "answer": answer, "time_seconds": elapsed}


print("Agent ready.")

## 3. Running All 7 Example Questions

In [ ]:
QUESTIONS = [
    "What is the capital of Austria, and what is its current population?",
    "Convert 100 EUR to USD.",
    "Search for recent information about Agentic AI and summarize it in three sentences.",
    "If 1 EUR equals X USD, how much is 250 EUR in USD?",
    "Find information about MCP and explain why it is useful for agent tools.",
    "Compare EUR to GBP and USD using live exchange-rate data.",
    "What is Tesla's current stock price?"
]

all_results = []
for q in QUESTIONS:
    result = run_agent(q)
    all_results.append(result)

In [ ]:
# Save results and print summary table
os.makedirs("results", exist_ok=True)
with open("results/agent_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("=" * 65)
print("SUMMARY TABLE")
print("=" * 65)
print(f"{'#':<3} {'Question':<46} {'Tool':<22} {'Time':>6}")
print("-" * 65)
for i, r in enumerate(all_results, 1):
    print(f"{i:<3} {r['question'][:44]:<46} {r['tool']:<22} {r['time_seconds']:>5.2f}s")
print()
print("Saved to results/agent_results.json")